# [2.3: Jaccard] // Content-based Filtering with Categorical Features

We're going to look at one more way to compute similarities. In the previous notebooks, you built similarity matrices based on textual content, using TF–IDF and Word2Vec representations. In all cases, the core idea was the same:
represent items as vectors and use a similarity measure to compare them. Once you have a similarity matrix you can use kNN to do recommendations.

However, not all data is textual or continuous. Many real-world recommender systems also rely on **categorical information**, such as topics, tags, regions, genres, or formats. Since these are not continuous values cosine similarity is not the best way to compute similarities with thos features.

In this notebook, you will see how **categorical features** can be used to construct a similarity matrix. The approach is deliberately simple:

* Categorical features are transformed into a **binary (one-hot) representation**
* Similarity between items is computed using the **Jaccard index**, which measures overlap between sets

The resulting similarity matrix plays the same role as in previous notebooks and could, in principle, be used in kNN-based recommenders or combined with rating data. We will not implement those steps here. The goal is simply to show that **similarity-based recommendation is not limited to text or numeric features**, but can also be built from structured categorical data.

## Data

The MIND dataset does not provide suitable categorical features for this example. Since our goal here is to **illustrate the underlying concepts**, we will just create a small, manually constructed dataset instead. Load below:

In [ ]:
import pandas as pd
import pooch

%reload_ext autoreload
%autoreload 2

DATA_REPO = "https://raw.githubusercontent.com/uvapl/recommender-systems/2025/data/m2/"
for fname in ["tests_m2.py"]:
    pooch.retrieve(url = DATA_REPO + fname, known_hash=None, fname=fname, path=".", progressbar=True)

import tests_m2

articles = pd.DataFrame({
    "article_id": ["N001", "N002", "N003", "N004", "N005"],
    "topics": [["politics", "europe"], ["politics", "economy"], ["sports", "football"], ["sports", "tennis"], ["economy", "climate"]],
    "regions": [["EU"], ["EU", "US"], ["EU"], ["global"], ["global"]],
    "formats": [["news"], ["analysis"], ["news"], ["interview"], ["analysis"]]
})

articles


## Feature representation

In the dataset, categorical features are stored as lists (e.g. `[EU, US]`, `[sports, tennis]`). While this representation is convenient for humans, it is not suitable as input for most machine learning algorithms. We therefore need to convert these list-based features into a one-hot encoded representation.

**One-hot encoding** represents each possible category as its own column.
For each article:

* The value is `1` (or `True`) if the feature is present
* The value is `0` (or `False`) if the feature is absent

This turns each article into a **binary feature vector** that can be used as input for our machine learning algorithm.

### Example

Suppose we have the following features for two articles:

    Article A: ["politics", "EU", "news"]
    Article B: ["politics", "US", "analysis"]

After one-hot encoding, we want the data to look like this:

| article_id | politics | EU | US | news | analysis |
| ---------- | -------- | -- | -- | ---- | -------- |
| A          | 1        | 1  | 0  | 1    | 0        |
| B          | 1        | 0  | 1  | 0    | 1        |

This representation allows us later to compute similarities between articles using set-based measures such as the Jaccard index.


### Question 1

*3 pts.*

Complete the `transform_one_hot()` function below.

The input is a DataFrame containing articles, where categorical features are stored as **lists**.
The function should return a **one-hot encoded DataFrame** with:

* **Rows** representing articles
* **Columns** representing individual features
* **Values** equal to `1` if the article has the feature, and `0` otherwise

In other words, each article should be represented as a binary feature vector.

In [ ]:
def transform_one_hot(articles):
    # your code here

one_hot = transform_one_hot(articles)
one_hot

In [ ]:
# test your solution

tests_m2.jaccard_test_01(one_hot)

## Jaccard index

The **Jaccard index** is a similarity measure that allows us to compare **categorical data**. It answers the question:

*“Between two articles, how many features do they have in common?”*

For example, consider two articles with the following feature sets:

```
Article A: {"politics", "EU", "news"}
Article B: {"politics", "US", "analysis"}
```

* Shared features (intersection): `{"politics"}`
  Number of shared features = 1

* Features present in at least one of the two articles (union):
  `{"politics", "EU", "US", "news", "analysis"}`
  Total number of features = 5

So the Jaccard similarity is:

$$
J = \frac{\text{intersection}}{\text{union}} = \frac{1}{5} = 0.2
$$

Intuitively, this means that **only a small fraction of the features used by either article are shared**.

---

### Formal definition

The Jaccard index is formally defined in terms of mathematical sets.
For two sets $A$ and $B$, the Jaccard index is:

$$
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

where:

* $A \cap B$ is the set of elements shared by both sets
* $A \cup B$ is the set of all elements that appear in either set

The Jaccard index always takes a value between `0` and `1`:

* `0` means no overlap at all
* `1` means the sets are identical

### Question 2

*3 pts.*

Implement `jaccard_index_matrix()` below. The function takes the one-hot encoded feature matrix `one_hot` as input and should return an article–article similarity matrix computed using the Jaccard index.

In [ ]:
# your code here

In [ ]:
# test your solution

tests_m2.jaccard_test_02(similarity)

### Conclusion

In this notebook, you saw how *categorical features* can be converted into a *one-hot encoded* representation and how the *Jaccard index* can be used to compute a similarity matrix.

In principle, this similarity matrix could be used as input for a *kNN-based recommender*: for a given article (or user profile), you would retrieve the most similar articles and recommend them. In this notebook we do not take that final step, because we do not have the user–article interaction data needed to generate recommendations and evaluate their performance.

That said, the approach is the same as before. You have already implemented kNN in two variants, *regression* and *classification*, and either version could be adapted to work with a categorical-feature similarity matrix like the one you built here.